In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

In [3]:
def get_products_for_rate_bases(products_path, rate_bases_path):
    with open(products_path, 'r') as f:
        products = json.load(f)
    
    with open(rate_bases_path, 'r') as f:
        rate_bases = json.load(f)

    valid_rate_base_codes = set(rb.get("code") for rb in rate_bases if "code" in rb)

    filtered_products = [
        p for p in products
        if p.get("rateBasis") in valid_rate_base_codes
    ]

    return filtered_products


def append_to_json_file(data, file):
    if os.path.exists(file):
        return 
    
    with open(file, 'w', encoding='utf-8') as json_file:
        json.dump(data, json_file, indent=4, ensure_ascii=False)
    print(f"Data saved to {file}")

In [2]:
def merged_products_with_ratebasis(products_data_path, ratebasis_data_path):
    """
    based on the ratebasis code, merge the rate basis data with the products data.
    """

    with open(products_data_path, 'r') as f:
        products_data = json.load(f)
    
    with open(ratebasis_data_path, 'r') as f:
        ratebasis_data = json.load(f)
    
    ratebasis_lookup = {
        rb['code'] : {
            'rate_code': rb['code'],
            'property': rb['property'],
            'booking_period': rb['booking_period'],
            'stay_period': rb['stay_period'],
            'sales_segments': rb['sales_segments'],
            'description': rb['description'],
            'commission': rb['commission'],
            'payOnArrival': rb['payOnArrival'],
            'currency': rb['currency'],
            'taxInc': rb['taxInc']
        } for rb in ratebasis_data
    }

    merged_data = []
    for product in products_data:
        if 'rateBasis' not in product:
            merged_data.append(product)
            continue

        rate_basis_code = product['rateBasis']
        if rate_basis_code in ratebasis_lookup:
            merged_item = dict(product)
            calendar_data = merged_item.pop('calendar', [])

            merged_item.update({
                'rate_code': ratebasis_lookup[rate_basis_code]['rate_code'],
                'property': ratebasis_lookup[rate_basis_code]['property'],
                'booking_period': ratebasis_lookup[rate_basis_code]['booking_period'],
                'stay_period': ratebasis_lookup[rate_basis_code]['stay_period'],
                'sales_segments': ratebasis_lookup[rate_basis_code]['sales_segments'],
                'rate_description': ratebasis_lookup[rate_basis_code]['description'],
                'commission': ratebasis_lookup[rate_basis_code]['commission'],
                'payOnArrival': ratebasis_lookup[rate_basis_code]['payOnArrival'],
                'rate_currency': ratebasis_lookup[rate_basis_code]['currency'],
                'taxInc': ratebasis_lookup[rate_basis_code]['taxInc']
            })

            merged_item['calendar'] = calendar_data
            merged_data.append(merged_item)
        else:
            merged_data.append(product)
            print(f"Rate basis not found: {rate_basis_code}")
    
    return merged_data

In [4]:
merged_products_data = merged_products_with_ratebasis('../Data/MainData/productsData.json', '../Data/MainData/rateBaseData.json')
append_to_json_file(merged_products_data, '../Data/TransformedData/ProductsRateBasisData.json') 

Data saved to ../Data/TransformedData/ProductsRateBasisData.json


In [5]:
def merged_allotments_data(allotmentsData_path, productsRateBasisData_path):
    """
    Merge the allotments data with the productsRateBasis data.
    """

    with open(allotmentsData_path, 'r') as f:
        allotments_data = json.load(f)
    
    with open(productsRateBasisData_path, 'r') as f:
        products_ratebasis = json.load(f)

    allotments_by_base = {}
    for allotment in allotments_data:
        if 'bases' in allotment and isinstance(allotment['bases'], list):
            for base in allotment['bases']:
                if base not in allotments_by_base:
                    allotments_by_base[base] = []
                allotments_info = {
                    'allotment_code': allotment['code'],
                    'allotment_unit': allotment['unit'],
                    'contracted': allotment.get('contracted'),
                    'allotment_createdOn': allotment.get('createdOn'),
                    'allotment_calendar': allotment.get('calendar', [])
                }
                allotments_by_base[base].append(allotments_info)

    merged_data = []
    for product in products_ratebasis:
        if 'rate_code' not in product:
            print(f"Rate code not found: {product.get('rate_code')}")
            continue

        rate_code = product['rate_code']
        matching_allotments = allotments_by_base.get(rate_code, [])

        if matching_allotments:

            for allot in matching_allotments:
            
                merged_item = dict(product)

                calendar_data = merged_item.pop('calendar', [])

                merged_item.update({
                    'allotment_code': allot['allotment_code'],
                    'allotment_unit': allot['allotment_unit'],
                    'contracted': allot['contracted'],
                    'allotment_createdOn': allot['allotment_createdOn'],
                    'allotment_calendar': allot['allotment_calendar']
                })

                merged_item['calendar'] = calendar_data

            merged_data.append(merged_item)
        else:
            product.update({
                'allotment_code': None,
                'allotment_unit': None,
                'contracted': None,
                'allotment_createdOn': None,
                'allotment_calendar': []
            })
            merged_data.append(product)
    
    return merged_data

In [6]:
merged_allotments = merged_allotments_data('../Data/MainData/allotmentsData.json', '../Data/TransformedData/ProductsRateBasisData.json')
append_to_json_file(merged_allotments, '../Data/TransformedData/ProductsRateBasisAllotmentsData.json')    

Data saved to ../Data/TransformedData/ProductsRateBasisAllotmentsData.json


In [7]:
def flatten_calendar_entries(meregd_data_path):
    """
    flatten the calendar entries in transformed data.
    """

    with open(meregd_data_path, 'r') as f:
        merged_data = json.load(f)
    
    flattened_data = []

    for entry in merged_data:
        product_calendar = entry.get('calendar', [])
        allotment_calendar = entry.get('allotment_calendar', [])

        base_entry = dict(entry)
        base_entry.pop('calendar', None)
        base_entry.pop('allotment_calendar', None)

        max_calendar_length = max(len(product_calendar), len(allotment_calendar))

        if max_calendar_length == 0:
            flat_entry = dict(base_entry)
            flat_entry.update({
                'allotment': None,
                'allotment_date': None,
                'releaseDate': None,
                'available': None,
                'booked': None,
                'sell': None,
                'product': None,
                'product_date': None,
                'public': None,
                'price': None,
                'supplement': None                
            })
            flattened_data.append(flat_entry)
            continue

        for i in range(max_calendar_length):
            flat_entry = dict(base_entry)

            if i < len(allotment_calendar):
                allotment_entry = allotment_calendar[i]
                flat_entry.update({
                    'allotment': entry.get('allotment_code'),
                    'allotment_date': allotment_entry.get('date'),
                    'releaseDate': allotment_entry.get('releaseDate'),
                    'available': allotment_entry.get('available'),
                    'booked': allotment_entry.get('booked'),
                    'sell': allotment_entry.get('sell')
                })
            else:
                flat_entry.update({
                    'allotment': None,
                    'allotment_date': None,
                    'releaseDate': None,
                    'available': None,
                    'booked': None,
                    'sell': None
                })
            
            if i < len(product_calendar):
                product_entry = product_calendar[i]
                flat_entry.update({
                    'product': entry.get('code'),
                    'product_date': product_entry.get('date'),
                    'public': product_entry.get('public'),
                    'price': product_entry.get('price'),
                    'supplement': product_entry.get('supplement')
                })
            else:
                flat_entry.update({
                    'product': None,
                    'product_date': None,
                    'public': None,
                    'price': None,
                    'supplement': None
                })
            flattened_data.append(flat_entry)

    return flattened_data

In [8]:
merged_data = flatten_calendar_entries('../Data/TransformedData/ProductsRateBasisAllotmentsData.json')
append_to_json_file(merged_data, '../Data/TransformedData/FlattenedCalendarData.json') #type: ignore

Data saved to ../Data/TransformedData/FlattenedCalendarData.json
